# 10j — single-time-point model diagnostics (origin 2020-11-08)

A **single-forecast-date** companion to `9j_forecast_diagnostics.ipynb` (which draws
period-wide diagnostics). **8j fits and caches** the MCMC chains to
`../dt_intermediate/8j_chn_*.jld2`; this notebook **reloads them** (no re-fit) for the one
origin **2020-11-08** and draws two diagnostics:

1. the **four ways** total-infection forecast point + 90% CI in one panel, overlaid on observed;
2. for the two **mean-NGM** models, the **observed vs GP-smoothed contact mean μ_{i→j}** by age
   pair, for participant groups **16-24** and **25-34**, over contactee age group.

The shared setup (`cfg`, `grid`, `raw`, `FORECAST_ORIGINS`, `wins`, `combos`) is reproduced
verbatim from 8j/9j so the chain-cache keys `(degree, ngm, contacts, origin, h)` match exactly.
Outputs go to `res/10j_*`.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

In [ ]:
# Same config as 8j/9j so the cache keys match. `constant_contacts = false` ⇒ contact degree
# estimated PER WEEK (independent age-pair GP each window week; shared ρ, η; per-week level cₜ +
# field zₜ). The cached chains this notebook reloads were fit under exactly this setting.
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

raw  = load_raw_contact_inputs()
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

# The single forecast date this notebook diagnoses (all four models cached for it, h1–h4).
ORIGIN = Date(2020, 11, 8)
@assert ORIGIN in FORECAST_ORIGINS "ORIGIN $(ORIGIN) not in available origins $(first(FORECAST_ORIGINS))…$(last(FORECAST_ORIGINS))"
win = wins[findfirst(==(ORIGIN), FORECAST_ORIGINS)]

println("origin           : ", ORIGIN)
println("fit weeks        : ", win.fit_weeks[1], " … ", win.fit_weeks[end])
println("forecast weeks   : ", win.forecast_weeks[1], " … ", win.forecast_weeks[end])

In [ ]:
# The four combos (Axis1 degree × Axis2 NGM), fixed order + colour palette; read-only chain viz.
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
labels4    = [string(degree_label(dm), "|", ngm_label(nb)) for (dm, nb) in combos]
model_cols = [:steelblue, :darkorange, :seagreen, :purple]

include("8j_viz_utils.jl")    # chain_path, _group_matrix, load_transmission_draws, …
include("10j_viz_utils.jl")   # reconstruct_mu_draws (smoothed μ per draw)
println("combos = ", labels4)

In [ ]:
# Reload the cached chains for THIS origin only and assemble the forecast fans (NO re-fit).
# `iterated_forecast` → `fit_or_load_chain` reloads each cached chain; a missing one would
# trigger a fallback re-fit (run 8j first). Keyed by model label (single origin).
wd    = load_window_data(win; grid = grid)
truth = load_forecast_truth(win; grid = grid)
# this origin's 4 contact/degree windows (one per horizon; reuse the single raw read)
apd_o = [prepare_degree_data(
             WeeklyWindow(ORIGIN + Day(7 * (h - 1));
                          n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
             cfg; grid = grid, setting = :all,
             df_part_raw = raw.df_part, craw_raw = raw.craw)
         for h in cfg.horizons]

fc_store = Dict{String,Array{Float64,3}}()
for (dm, nb) in combos
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    try
        fc_store[lbl] = iterated_forecast(dm, nb, wd, cfg, win;
                                          grid = grid, setting = :all, use_nuts = USE_NUTS,
                                          save_dir = "../dt_intermediate", apd_by_h = apd_o)
    catch err
        @warn "skipped combo (missing/pathological chain)" model=lbl exception=err
    end
end
println("assembled fans for: ", collect(keys(fc_store)))

## 1. Forecast point + 90% CI — four ways, single origin

One panel: observed weekly infections (8 fit weeks of history ++ the 4 realised target weeks,
all ages summed) overlaid with each of the four models' total-infection forecast (median + 90%
band). Dashed line marks the origin.

In [ ]:
qs_lo, qs_hi = 0.05, 0.95
H = length(cfg.horizons)

x_hist = week_mid.(win.fit_weeks)
y_hist = vec(sum(wd.I_mean[:, (cfg.smax + 1):end]; dims = 1))    # history (all ages)
x_fore = week_mid.(win.forecast_weeks)
y_fore = [sum(truth[:, h]) for h in 1:H]                         # realised targets (all ages)

fc_fig = plot(; title = "10j — total-infection forecast (four ways) vs observed, origin $(ORIGIN) (90% band)",
              titlefontsize = 9, xrotation = 45, legend = :topleft, size = (950, 540),
              xlabel = "week (Wed mid-date)", ylabel = "weekly infections (all ages)")
plot!(fc_fig, vcat(x_hist, x_fore), vcat(y_hist, y_fore);
      color = :black, lw = 2, marker = :circle, ms = 3, label = "observed")
vline!(fc_fig, [week_mid(win.origin)]; color = :gray, ls = :dash, lw = 1, label = "")
for (ci, lbl) in enumerate(labels4)
    haskey(fc_store, lbl) || continue
    tot = dropdims(sum(fc_store[lbl]; dims = 1); dims = 1)       # H × draws (age-summed)
    med = [median(tot[h, :]) for h in 1:H]
    lo  = [quantile(tot[h, :], qs_lo) for h in 1:H]
    hi  = [quantile(tot[h, :], qs_hi) for h in 1:H]
    plot!(fc_fig, x_fore, med; color = model_cols[ci], lw = 1.8, marker = :circle, ms = 2,
          ribbon = (med .- lo, hi .- med), fillalpha = 0.12, label = lbl)
end
savefig(fc_fig, "../res/10j_forecast_ci_$(ORIGIN).png")
fc_fig

## 2. Observed vs GP-smoothed contact mean μ by age pair (mean-NGM models)

For the two **mean-NGM** models, the raw GP-smoothed directional contact mean **μ_{i→j}**
(shown *as-is* — the smoothing part, no (1−p⁰) factor) is reconstructed per posterior draw from
the origin-week slice of the h=1 chain, and compared with the matching observed empirical mean:

- **unweighted-negbin | mean** — μ is the per-capita count mean; observed = `apd.emp_mean`.
- **weighted-hweibull | mean** — μ is the Weibull scale = mean of *positive* duration-weighted
  degrees; observed = `whist_mean(apd.pos_weight)` (collapsed histogram).

Each model → a 2×1 figure overlaying every participant age group, coloured per participant:
**upper = young participants (2-34)**, **lower = older (35+)**; x-axis = contactee age group.
Lines are the smoothed μ (median + 90% ribbon); `×` markers are the observed means.

In [ ]:
# Observed degree data at the origin window (last week == origin). Weekly regime keeps [t,i,j].
apd = prepare_degree_data(WeeklyWindow(ORIGIN; n_fit = cfg.n_fit, smax = cfg.smax,
                                       horizons = cfg.horizons),
                          cfg; grid = grid, setting = :all,
                          df_part_raw = raw.df_part, craw_raw = raw.craw)
t_o = length(apd.weeks)                          # origin week = last of all_weeks
@assert apd.weeks[t_o] == win.origin

# Participants split across two stacked panels: upper = young (2-34), lower = older (35+).
const PART_ROWS = ((1, 2, 3, 4), (5, 6, 7))
const ROW_TITLE = ("participants 2-34", "participants 35+")
const PART_COLS = [:steelblue, :darkorange, :seagreen, :purple, :crimson, :goldenrod, :teal]

# One panel: overlay every participant in `parts` — smoothed μ (line + 90% ribbon) and the
# matching observed mean (× markers), coloured by participant; x-axis = contactee age group.
function agepair_panel(parts, ttl, μdraws, weighted)
    pnl = plot(; title = "$(ttl)  (lines = smoothed μ, 90%;  × = observed)", titlefontsize = 8,
               xticks = (1:grid.N, grid.LAB), xrotation = 45,
               xlabel = "contactee age group", ylabel = "contact mean μ",
               legend = :topright, legendfontsize = 6, legendtitle = "participant",
               legendtitlefontsize = 6)
    for i in parts
        obs = Float64[]
        for j in 1:grid.N
            if weighted
                pw = apd.pos_weight[t_o, i, j]
                push!(obs, isempty(pw) ? NaN : whist_mean(pw))   # positive duration-weighted mean (histogram)
            else
                push!(obs, apd.emp_mean[t_o, i, j])          # per-capita count mean
            end
        end
        med = [median(μdraws[:, i, j]) for j in 1:grid.N]
        lo  = [quantile(μdraws[:, i, j], 0.05) for j in 1:grid.N]
        hi  = [quantile(μdraws[:, i, j], 0.95) for j in 1:grid.N]
        c = PART_COLS[i]
        plot!(pnl, 1:grid.N, med; ribbon = (med .- lo, hi .- med), color = c, lw = 2,
              marker = :circle, ms = 3, fillalpha = 0.08, label = grid.LAB[i])
        scatter!(pnl, 1:grid.N, obs; color = c, marker = :x, ms = 5, msw = 2, label = "")
    end
    return pnl
end

# Build the 2×1 (upper young / lower older) observed-vs-smoothed-μ figure for one mean-NGM model.
function make_agepair_fig(dm::ContactDegreeModel, nb::NGMBuilder)
    lbl = string(degree_label(dm), "|", ngm_label(nb))
    μdraws = reconstruct_mu_draws(lbl, ORIGIN, 1; week_index = t_o, grid = grid)
    μdraws === nothing && (@warn "no chain for $lbl @ $ORIGIN"; return nothing)
    weighted = is_weighted(dm)
    panels = [agepair_panel(PART_ROWS[r], ROW_TITLE[r], μdraws, weighted) for r in 1:length(PART_ROWS)]
    scale_note = weighted ? "positive duration-weighted degree" : "per-capita count"
    fig = plot(panels...; layout = (2, 1), size = (760, 820),
               plot_title = "$(lbl) — observed vs smoothed μ ($(scale_note)), origin $(ORIGIN)",
               plot_titlefontsize = 10)
    fname = weighted ? "../res/10j_agepair_mean_hweibull_$(ORIGIN).png" :
                       "../res/10j_agepair_mean_negbin_$(ORIGIN).png"
    savefig(fig, fname)
    return fig
end

In [ ]:
# unweighted-negbin | mean — smoothed μ is the per-capita count mean (observed = emp_mean)
make_agepair_fig(NegBinAgePair(), MeanNGM())

In [ ]:
# weighted-hweibull | mean — smoothed μ is the positive-weight mean (observed = whist_mean(pos_weight))
make_agepair_fig(HurdleWeibullAgePair(), MeanNGM())

## Notes

- **Single origin** (2020-11-08); no re-fit — cached 8j chains reloaded via `iterated_forecast`
  (fans) and `reconstruct_mu_draws` (smoothed μ).
- **§2 shows the two mean-NGM models only** (neighbourhood NGM changes the C* functional, not the
  smoothed μ, so it adds nothing to this contact-mean view).
- **μ is shown raw** on each model's own scale (the "smoothing part"): negbin μ = per-capita count
  mean; hweibull μ = mean of positive duration-weighted degrees. No (1−p⁰) per-capita rescaling.
- The smoothed μ is the **origin-week** slice (`week_index = t_o`, the last window week) of the
  **h=1** chain — the week the forecast NGM is frozen at.
- All 7 participant age groups shown, overlaid and coloured; split upper (2-34) / lower (35+);
  x-axis = contactee age group. Lines = smoothed μ (90% ribbon), × = observed.